# Download Earthquake hypocenters from USGS

This is an example notebook of how to download Earthquake data from the USGS catalog programatically into a CSV and then convert to a VTK for viewing in Paraview and shapefile for GIS software.

**Note**: the download can take some time so be patient.

You will need a few packages:

- **libcomcat**:   `pip install usgs-libcomcat`
- **pyevtk**:      `pip install pyevtk` 



In [18]:
from datetime import datetime
from libcomcat.search import search
from libcomcat.dataframes import get_summary_data_frame

import geopandas as gpd
from pyevtk.hl import pointsToVTK

# Search Parameters

Here you ca set the start time and end time of earthquakes, a bounding box, magnitude, etc see [usgs-libcomcat docs](https://code.usgs.gov/ghsc/esi/libcomcat-python/-/blob/main/docs/api.md?ref_type=heads) for more details.

In [3]:
box_events = search(
    starttime=datetime(1960, 1, 1),
    endtime=datetime(2020, 12, 31),
    minlatitude=37.612,
    maxlatitude=38.425,
    minlongitude=-118.3,
    maxlongitude=-117.216,
    minmagnitude=0.0,
)

## Download Earthquake data
This part may take some time, the search is not optimized for speed.  The result will be a `pandas.DataFrame`.

In [6]:
eq_df = get_summary_data_frame(box_events)

### Convert to Shapefile

In [11]:
eq_gdf = gpd.GeoDataFrame(
    eq_df,  geometry=gpd.points_from_xy(eq_df.longitude, eq_df.latitude),
    crs="EPSG:4326"
)

In [12]:
eq_gdf.to_file("cmcv_eq_all.shp")

C:\Users\jpeacock\AppData\Local\Temp\1\ipykernel_23556\4064314202.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  eq_gdf.to_file("cmcv_eq_all.shp")
c:\Users\jpeacock\AppData\Local\miniforge3\envs\py313\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(
c:\Users\jpeacock\AppData\Local\miniforge3\envs\py313\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'significance' to 'significan'
  ogr_write(


### Convert to VTK

Project onto the models EPSG

In [14]:
eq_gdf_utm = eq_gdf.to_crs("EPSG:32611")

In [21]:
pointsToVTK(
    "cmcv_eq_all",
    eq_gdf_utm.geometry.x.to_numpy() / 1000,
    eq_gdf_utm.geometry.y.to_numpy() / 1000,
    eq_gdf_utm.depth.to_numpy() * -1,
    data={
        "magnitude": eq_gdf_utm.magnitude.to_numpy(),
        "depth": eq_gdf_utm.depth.to_numpy(),
        "time": eq_gdf_utm.time.astype(int).to_numpy(),
    },
)

'c:\\Users\\jpeacock\\OneDrive - DOI\\Documents\\GitHub\\sandbox_scripts\\cmcv_eq_all.vtu'